# Module 2 — Variable Scope & Namespaces
A **namespace** maps names→objects. **Scope** is where a namespace is reachable. Python resolves names with the **LEGB** rule: Local → Enclosing → Global → Built-in.

## 1. LEGB in action
The same name can exist at several levels; Python picks the nearest.

In [1]:
x = 'global-x'

def outer():
    x = 'enclosing-x'
    def inner():
        x = 'local-x'
        print('inner sees:', x)      # Local
    inner()
    print('outer sees:', x)          # Enclosing/local of outer

outer()
print('module sees:', x)             # Global
print('built-in example, len:', len([1, 2, 3]))   # Built-in

inner sees: local-x
outer sees: enclosing-x
module sees: global-x
built-in example, len: 3


## 2. The UnboundLocalError trap
Assigning to a name anywhere in a function makes it **local** for the whole function — so reading it first fails.

In [2]:
count = 0

def buggy_increment():
    count = count + 1     # 'count' is local because we assign to it
    return count

try:
    buggy_increment()
except UnboundLocalError as e:
    print('UnboundLocalError:', e)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


### Fix A — `global` (rebind the module variable)

In [3]:
count = 0
def increment():
    global count
    count += 1
    return count

increment(); increment()
print('count is now', count)

count is now 2


### Fix B — return a value (cleaner, no global state)

In [4]:
def increment(c):
    return c + 1

c = 0
c = increment(c)
c = increment(c)
print('c is now', c)

c is now 2


## 3. Enclosing scope, closures & `nonlocal`
A nested function remembers variables from its enclosing function — that's a **closure**. `nonlocal` lets it rebind them.

In [5]:
def make_counter():
    n = 0
    def step():
        nonlocal n        # rebind the enclosing n
        n += 1
        return n
    return step           # the closure keeps 'n' alive

c = make_counter()
print(c(), c(), c())       # 1 2 3 — state remembered between calls

def make_multiplier(k):
    return lambda x: x * k  # closure over k

triple = make_multiplier(3)
print('triple(10) =', triple(10))

1 2 3
triple(10) = 30


## 4. Inspecting namespaces

In [6]:
import math
y = 42
def show():
    local_only = 7
    print('locals():', locals())
show()
print("'y' in globals():", 'y' in globals())
print("'pi' in dir(math):", 'pi' in dir(math))
print('a few built-ins:', [n for n in dir(__builtins__) if n.islower()][:6])

locals(): {'local_only': 7}
'y' in globals(): True
'pi' in dir(math): True
a few built-ins: ['__build_class__', '__debug__', '__doc__', '__import__', '__loader__', '__name__']


## 5. Two classic pitfalls (and fixes)

In [7]:
# (a) Mutable default argument — created ONCE, shared across calls
def add_item_bug(item, bag=[]):
    bag.append(item); return bag

print('bug:', add_item_bug('a'), add_item_bug('b'))   # ['a','b'] — leaked!

def add_item_ok(item, bag=None):
    if bag is None:
        bag = []
    bag.append(item); return bag

print('ok :', add_item_ok('a'), add_item_ok('b'))     # ['a'] ['b']

bug: ['a', 'b'] ['a', 'b']
ok : ['a'] ['b']


In [8]:
# (b) Late-binding closures in a loop
funcs_bug = [lambda: i for i in range(3)]
print('bug:', [f() for f in funcs_bug])      # 2 2 2 — all see final i

funcs_ok = [lambda i=i: i for i in range(3)]  # capture current i as a default
print('ok :', [f() for f in funcs_ok])        # 0 1 2

bug: [2, 2, 2]
ok : [0, 1, 2]


## 6. Practice exercises

1. Predict, then verify, the output of a function that reads a global without assigning.
2. Write `make_accumulator()` that returns a function adding to a running total (use `nonlocal`).
3. Add a `reset()` capability to `make_counter` (hint: return two functions).
4. Demonstrate shadowing by naming a variable `sum` and then calling the built-in `sum`.


In [9]:
# Your turn:
